# 14 · 归一化：BatchNorm 与 LayerNorm

> **本节属于 Part 5 · 优化与训练工程。**

深层网络训练时，每一层输入的分布会随训练不断漂移，拖慢收敛。**归一化层**通过把激活拉回"零均值、单位方差"再做可学习的缩放平移，显著**加速并稳定**训练。本节实现 **BatchNorm** 与 **LayerNorm**，并手推 BatchNorm 那个著名（且复杂）的反向公式——然后见证 autograd 自动把它算对。

## 学习目标

- 理解 BatchNorm / LayerNorm 的前向：标准化 + 可学习的 $\gamma, \beta$
- 理解训练/推理两种模式（BatchNorm 的"运行统计量"）
- **手推 BatchNorm 的反向公式**，并用 gradcheck 验证 autograd 与它一致
- 看到归一化对训练速度的实际提升

## 前向：标准化 + 仿射

对 BatchNorm，在一个 batch 内**按特征**算均值方差：

$$\mu = \frac{1}{N}\sum_i x_i,\quad \sigma^2 = \frac{1}{N}\sum_i (x_i-\mu)^2,\quad \hat{x} = \frac{x-\mu}{\sqrt{\sigma^2+\epsilon}},\quad y = \gamma\hat{x} + \beta$$

LayerNorm 几乎一样，只是把"按特征跨样本"换成"按样本跨特征"（在最后一维上归一化）——这是 Transformer 的标配。看实现：

In [ ]:
import inspect
import numpy as np
import matplotlib.pyplot as plt
import minitorch
from minitorch import Tensor, nn, no_grad, data
from minitorch.optim import Adam
from minitorch.utils import numerical_gradient, rel_error

print(inspect.getsource(nn.BatchNorm1d.forward))

## 反向：复杂公式，autograd 却"免费"算对

BatchNorm 的反向公式以"难推"著称。设上游梯度为 $\frac{\partial L}{\partial y}$，记 $\hat{x}$ 为标准化值、$N$ 为 batch 大小，则对输入的梯度是：

$$\frac{\partial L}{\partial x_i} = \frac{\gamma}{\sqrt{\sigma^2+\epsilon}}\cdot\frac{1}{N}\Big(N\frac{\partial L}{\partial \hat{x}_i} - \sum_j \frac{\partial L}{\partial \hat{x}_j} - \hat{x}_i \sum_j \frac{\partial L}{\partial \hat{x}_j}\hat{x}_j\Big)$$

**好消息是：我们一行都不用手写这个公式！** 因为前向是用 `Tensor` 算子（`mean`、`-`、`/`、`**0.5`…）搭出来的，autograd 会自动、正确地反传。下面用数值梯度检查来**证明** autograd 算出的就是上面这个复杂公式的结果：

In [ ]:
np.random.seed(0)
x_np = np.random.randn(16, 4) * 2 + 1
bn = nn.BatchNorm1d(4); bn.train()

R = np.random.randn(16, 4)                  # 随机加权和作为标量损失
x = Tensor(x_np)
(bn(x) * Tensor(R)).sum().backward()

def loss_np(xv):
    mean = xv.mean(0); var = ((xv - mean) ** 2).mean(0)
    xhat = (xv - mean) / np.sqrt(var + bn.eps)
    return ((xhat * bn.gamma.data + bn.beta.data) * R).sum()

g = numerical_gradient(loss_np, x_np.copy())
print("autograd 算出的 BN 反向 vs 数值梯度，相对误差:", rel_error(x.grad, g))
print("=> autograd 自动算出了那个复杂公式，且完全正确 ✅")

## 训练/推理两种模式

训练时用**当前 batch** 的统计量，同时维护"运行统计量"（滑动平均）；推理时改用运行统计量（因为推理可能一次只来一个样本，没法算 batch 统计）。所以 BatchNorm 也必须区分 `train()/eval()`。

In [ ]:
bn = nn.BatchNorm1d(3)
x = Tensor(np.random.randn(8, 3) * 5 + 2)
bn.train(); out = bn(x)
print("train 模式：输出按特征近似零均值单位方差")
print("  mean:", np.round(out.data.mean(0), 3), " std:", np.round(out.data.std(0), 3))
print("  运行均值已更新:", np.round(bn.running_mean, 3))

## BatchNorm 加速训练

在 MNIST 上对比"有 BN / 无 BN"的收敛速度——BN 通常让损失下降得更快。

In [ ]:
(X_tr, y_tr), (X_te, y_te) = minitorch.utils.load_mnist(n_train=10000, n_test=2000)
loader = data.DataLoader(data.TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
loss_fn = nn.CrossEntropyLoss()

def train(model, epochs=5):
    opt = Adam(model.parameters(), lr=1e-3)
    losses = []
    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            L = loss_fn(model(Tensor(xb)), yb)
            L.backward(); opt.step()
            losses.append(float(L.data))
    return losses

minitorch.set_seed(0)
plain = train(nn.Sequential(nn.Linear(784,128), nn.ReLU(), nn.Linear(128,128), nn.ReLU(), nn.Linear(128,10)))
minitorch.set_seed(0)
withbn = train(nn.Sequential(nn.Linear(784,128), nn.BatchNorm1d(128), nn.ReLU(),
                             nn.Linear(128,128), nn.BatchNorm1d(128), nn.ReLU(), nn.Linear(128,10)))

plt.figure(figsize=(6,3.5))
def smooth(a, k=20): return np.convolve(a, np.ones(k)/k, mode="valid")
plt.plot(smooth(plain), label="no BN")
plt.plot(smooth(withbn), label="with BN")
plt.legend(); plt.xlabel("iteration"); plt.ylabel("loss (smoothed)")
plt.title("BatchNorm speeds up training"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f"前若干步平均损失：no BN={np.mean(plain[:100]):.3f}, with BN={np.mean(withbn[:100]):.3f}")

## PyTorch 对照

`nn.BatchNorm1d / nn.LayerNorm` 行为一致（注意 eval 模式下用运行统计量）。LayerNorm 在 Transformer 里无处不在，我们 Part 8 会大量用到它。

In [ ]:
ln = nn.LayerNorm(5)
x = Tensor(np.random.randn(3, 5))
out = ln(x)
print("LayerNorm 在最后一维归一化，各行 mean≈0 std≈1:")
print("  行 mean:", np.round(out.data.mean(-1), 3), " 行 std:", np.round(out.data.std(-1), 3))

## 📦 沉淀进 minitorch

`BatchNorm1d / LayerNorm` 在 `minitorch/nn/norm.py`，由 `tests/test_layers.py` 的梯度检查守护。**关键收获**：复杂层的反向不必手推——把前向用 Tensor 算子写对，autograd 就把反向送给你。

## 小练习

1. **手推验证**：用上面给出的 BatchNorm 反向公式，**手动**实现一遍 `dx`，与 autograd 的 `x.grad` 对比，确认一致。
2. **eval 的坑**：训练后不切 `eval()` 直接对单个样本推理会出问题（batch 统计量无意义），试一下并解释。
3. **LayerNorm vs BatchNorm**：为什么序列模型/Transformer 偏爱 LayerNorm 而非 BatchNorm？（提示：与 batch 内样本是否独立、序列长度可变有关。）

## 小结 & 下一站

✅ 我们实现了 BatchNorm/LayerNorm，理解了训练/推理两态，并见证 autograd 自动算对了 BatchNorm 那个"魔鬼"反向公式。**Part 5（优化与训练工程）至此完成**——minitorch 已具备训练现代网络的全套工具。

**下一站 → Part 6 `15_conv_intuition_naive`**：进入**卷积神经网络**！我们先用朴素循环把 2D 卷积写明白，再用 im2col 加速，最终训练一个 CNN 把 MNIST 准确率推到 98%+。